In [112]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import  Pipeline
import pandas as pd

In [99]:
df = pd.read_csv("HR_comma_sep.csv")
X = df.drop(columns=["left", "last_evaluation", "number_project", "time_spend_company"])
y = df.left
X.head(3)

,satisfaction_level,average_montly_hours,Work_accident,promotion_last_5years,Department,salary
0,0.38,157,0,0,sales,low
1,0.80,262,0,0,sales,medium
2,0.11,272,0,0,sales,medium


In [100]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [101]:
def get_score(model_name, X_train, X_test, y_train, y_test):
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), ("Department", "salary"))],
        remainder="passthrough"
    )
    model = Pipeline([
        ("pre", pre),
        ("reg", model_name),
    ])
    model.fit(X_train, y_train)
    return model.score(X_test, y_test)

In [102]:
get_score(LogisticRegression(max_iter=1000), X_train, X_test, y_train, y_test)

0.7833333333333333

In [103]:
get_score(SVC(), X_train, X_test, y_train, y_test)

0.7753333333333333

In [104]:
get_score(RandomForestClassifier(n_estimators=50), X_train, X_test, y_train, y_test)

0.9533333333333334

StratifiedKFold Class

In [105]:
fold = StratifiedKFold(n_splits=3)
scores_logistic = []
scores_svm = []
scores_rf = []

for train_index, test_index in fold.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    scores_logistic.append(get_score(LogisticRegression(solver='liblinear'), X_train, X_test, y_train, y_test))
    scores_svm.append(get_score(SVC(gamma='auto'), X_train, X_test, y_train, y_test))
    scores_rf.append(get_score(RandomForestClassifier(n_estimators=40), X_train, X_test, y_train, y_test))


In [106]:
scores_logistic

[0.7774, 0.7818, 0.7781556311262252]

In [107]:
scores_svm

[0.8506, 0.8402, 0.844368873774755]

In [108]:
scores_rf

[0.9412, 0.9264, 0.9661932386477295]

Cross_val_score


In [109]:
pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Department", "salary"])],
        remainder="passthrough"
    )
lr_model = Pipeline([
        ("pre", pre),
        ("reg", LogisticRegression(max_iter=1000)),
    ])

In [113]:
cross_val_score(lr_model, X, y, cv=4)

array([0.95786667, 0.89306667, 0.9528    , 0.97252601])

In [116]:
svc_model = Pipeline([
        ("pre", pre),
        ("reg", SVC()),
    ])

In [118]:
cross_val_score(svc_model, X, y, cv=4)

array([0.76906667, 0.77333333, 0.77226667, 0.77460656])

In [121]:
ran_model = Pipeline([
        ("pre", pre),
        ("reg", RandomForestClassifier(n_estimators=50)),
    ])

In [122]:
cross_val_score(ran_model, X, y, cv=4)

array([0.95866667, 0.8936    , 0.9568    , 0.97172579])